### G0-shift robustness experiment

In [1]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
# register and use Arial font in Matplotlib figures
font_manager.fontManager.addfont('ARIAL.TTF')
# Set global plotting defaults
plt.rcParams.update({
    'font.size': 20,
    'font.family': 'Arial',
    'xtick.labelsize': 20,
    'ytick.labelsize': 20
})

In [2]:
# Set random seeds for reproducibility 
import glob
import numpy as np
import random
import torch
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

##### Data Loading

In [3]:
# Load CuP traces from .npy files and split them into train/val/test subsets with a fixed random seed
def load_and_split_traces(npy_path, label, seed=42):
    data = np.load(npy_path, allow_pickle=True)  
    traces = list(data)
    
    # Reproducible shuffling
    random.seed(seed)
    random.shuffle(traces)
    n_total = len(traces)
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.15)
    n_test = n_total - n_train - n_val
    train_samples = [(t, label) for t in traces[:n_train]]
    val_samples   = [(t, label) for t in traces[n_train:n_train + n_val]]
    test_samples  = [(t, label) for t in traces[n_train + n_val:]]
    return train_samples, val_samples, test_samples

cup_paths = glob.glob("../data/CuP-CAF/CuP-0.1mM_train_val_test.npy")
cup_train_samples = []
cup_val_samples = []
cup_test_samples = []

for path in cup_paths:
    # Label=0 for CuP class; use the same seed to keep splits consistent across runs
    train_part, val_part, test_part = load_and_split_traces(path, label=0, seed=42)
    cup_train_samples.extend(train_part)
    cup_val_samples.extend(val_part)
    cup_test_samples.extend(test_part)
    
# Quick sanity check of split sizes
print(f"CuP training trace count: {len(cup_train_samples)}")
print(f"CuP validation trace count: {len(cup_val_samples)}")
print(f"CuP test trace count: {len(cup_test_samples)}")

CuP training trace count: 2325
CuP validation trace count: 498
CuP test trace count: 499


In [4]:
# Load CuP+CAF traces from .npy files, split them into train/val/test subsets, 
# and merge with the CuP (negative) samples.
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os

caf_paths = glob.glob("../data/CuP-CAF/CuP+CAF-1_2(0.2mM)_train_val.npy")
caf_train_samples = []
caf_val_samples = []

for path in caf_paths:
    data = np.load(path, allow_pickle=True)
    traces = list(data)  
    # Reproducible shuffle before splitting
    random.seed(42)
    random.shuffle(traces)
    n_total = len(traces)
    n_train = int(n_total * 0.8)
    n_val = n_total - n_train
    caf_train_samples.extend([(t, 1) for t in traces[:n_train]])
    caf_val_samples.extend([(t, 1) for t in traces[n_train:]])

# Merge positives with CuP negatives to form final train/val sets    
train_samples = cup_train_samples + caf_train_samples
val_samples   = cup_val_samples   + caf_val_samples

# Quick sanity check of split sizes
print(f"CAF training trace count: {len(caf_train_samples)}")
print(f"CAF validation trace count: {len(caf_val_samples)}")

CAF training trace count: 2217
CAF validation trace count: 555


In [5]:
# Dataset wrapper that converts each 1D conductance trace into paired time-domain 
# and frequency-domain (FFT magnitude) inputs.
class TraceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        trace, label = self.samples[idx]
        trace = np.array(trace, dtype=np.float32)
        # Time-domain input
        x_time = torch.tensor(trace).unsqueeze(0)  
        # Frequency-domain input
        freq = np.fft.rfft(trace)
        mag = np.abs(freq)
        x_freq = torch.tensor(mag[np.newaxis, :], dtype=torch.float32)

        return x_time, x_freq, label

##### Model

In [6]:
import torch.nn as nn
import torch.nn.functional as F

class TFC(nn.Module):
    def __init__(self, configs):
        super(TFC, self).__init__()
        
        # Time-domain encoder
        self.encoder_time = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=9, padding=4),
            nn.InstanceNorm1d(64, affine=True),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            
            nn.Flatten(),
            nn.Linear(128 * 750, 128),

            nn.LayerNorm(128), 
            nn.GELU(),
            nn.Dropout(0.25)
        )
        
        # Frequency-domain encoder
        self.encoder_freq = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.InstanceNorm1d(64, affine=True),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.InstanceNorm1d(128, affine=True),
            nn.GELU(),
            
            nn.Flatten(),
            nn.Linear(128 * 375, 128),
                
            nn.LayerNorm(128), 
            nn.GELU(),
            nn.Dropout(0.25)
        )

        # Projection heads
        self.projector_t = nn.Sequential(
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU()
        )
        
        self.projector_f = nn.Sequential(
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU()
        )

    def forward(self, x_in_t, x_in_f, mode="both"):
        # Return encoder features (h_*) and projected features (z_*)
        h_t, h_f = None, None
        z_t, z_f = None, None

        # Compute time branch if requested
        if mode in ["both", "time"]:
            h_t = self.encoder_time(x_in_t)  
            z_t = self.projector_t(h_t)       

        # Compute frequency branch if requested    
        if mode in ["both", "freq"]:
            h_f = self.encoder_freq(x_in_f)   
            z_f = self.projector_f(h_f)        

        return h_t, h_f, z_t, z_f

# classifier
class target_classifier(nn.Module):
    def __init__(self, configs):
        super(target_classifier, self).__init__()
        # classifier on concatenated (time, freq) features
        self.logits = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),       
            nn.Linear(128 * 2, 64), 
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, h_t, h_f):
         # In time-domain mode, use time-domain features
         # In frequency-domain mode, use frequency-domain features
         # In time-frequency fusion mode, use both time-domain and frequency-domain features
        if h_t is None: h_t = torch.zeros_like(h_f)
        if h_f is None: h_f = torch.zeros_like(h_t)
        
        h = torch.cat([h_t, h_f], dim=1) 
        out = self.logits(h)
        return out

##### Model Training (load model)

In [7]:
# Basic configs and device setup
class Config:
    TSlength_aligned_t = 1500
    TSlength_aligned_f = 751
    num_classes_target = 2

configs = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build model + classifier and load checkpoint weights
tfc_model = TFC(configs).to(device)
classifier = target_classifier(configs).to(device)

checkpoint = torch.load("checkpoint_both_(CuP+CAF).pth")
tfc_model.load_state_dict(checkpoint['tfc_model_state_dict'])
classifier.load_state_dict(checkpoint['classifier_state_dict'])

# Switch to evaluation mode
tfc_model.eval()
classifier.eval()

target_classifier(
  (logits): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.25, inplace=False)
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [8]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay,classification_report,accuracy_score
from mpl_toolkits.axes_grid1 import make_axes_locatable
from collections import Counter
import matplotlib.colors as mcolors

# Evaluate the model
def evaluate(tfc_model, classifier, dataloader, mode="both", return_acc=False, save_path=None):
    tfc_model.eval()
    classifier.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x_t, x_f, y in dataloader:
            x_t, x_f = x_t.to(device), x_f.to(device)
            h_t, h_f, z_t, z_f = tfc_model(x_t, x_f, mode=mode)
            logits = classifier(h_t, h_f)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
            
    # Metrics summary
    print(f"\n[Mode={mode}] Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))
    acc = accuracy_score(all_labels, all_preds)
    print(f"[Mode={mode}] Accuracy: {acc:.4f}")

    # Plot confusion matrix only when meaningful (>=2 classes and not perfect accuracy)
    unique_labels = np.unique(all_labels)
    if (acc < 1.0) and (len(unique_labels) >= 2):
        cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

        with np.errstate(divide='ignore', invalid='ignore'):
            cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm_normalized = np.nan_to_num(cm_normalized)

        display_labels = ["CuP", "CuP+CAF"]
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Custom colormap for better contrast
        colors = ["#FFFFFF", "#F3A697", "#F17D65"]
        cmap_custom = mcolors.LinearSegmentedColormap.from_list("white_to_red", colors)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized,
                                      display_labels=display_labels)
        disp.plot(cmap=cmap_custom, values_format='.3f', ax=ax, colorbar=False)
        
        # Colorbar and axis styling
        im = disp.im_
        im.set_clim(0, 1)

        cbar = fig.colorbar(im, ax=ax)
        cbar.ax.tick_params(labelsize=20)
        cbar.ax.yaxis.label.set_size(22)

        plt.setp(ax.get_yticklabels(), ha="right", multialignment="center", fontsize=20)
        ax.set_xlabel("Predicted label", fontsize=22, fontweight='normal')
        ax.set_ylabel("True label", fontsize=22, fontweight='normal')
        plt.setp(ax.get_xticklabels(), ha='center', fontsize=20)
        plt.setp(ax.get_yticklabels(), rotation=90, va='center', fontsize=20)

        for text in disp.ax_.texts:
            text.set_fontsize(22)
            val = disp.confusion_matrix[int(text.get_position()[1])][int(text.get_position()[0])]
            text.set_color('white' if val > 0.5 else 'black')

        plt.tight_layout()

        # Save figure if requested
        if save_path:
            dir_name = os.path.dirname(save_path)
            if dir_name:
                os.makedirs(dir_name, exist_ok=True)
            plt.savefig(save_path, dpi=600, bbox_inches='tight')

        plt.show()

    else:
        if len(unique_labels) < 2:
            print("Only one true class is present; skipping confusion matrix plotting.")
        else:
            print("Accuracy is 100%; skipping confusion matrix plotting.")

    if return_acc:
        return acc

    
# Report the predicted class distribution over a dataloader
def predict_class_distribution(tfc_model, classifier, dataloader, device):
    tfc_model.eval()
    classifier.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_t, x_f, y in dataloader:
            x_t, x_f = x_t.to(device), x_f.to(device)
            h_t, h_f, z_t, z_f = tfc_model(x_t, x_f, mode=mode)
            logits = classifier(h_t, h_f)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    counter = Counter(all_preds)
    total = sum(counter.values())
    for cls_idx in sorted(counter.keys()):
        count = counter[cls_idx]
        percent = 100 * count / total
    return counter

In [9]:
# Load traces from multiple .npy files and attach a class label to each trace.
# Return a list of (trace, label) pairs.
def load_trace_level_samples(file_label_pairs):
    samples = []
    for path, label in file_label_pairs:
        data = np.load(path, allow_pickle=True) 
        for trace in data:
            trace = np.array(trace, dtype=np.float32)
            samples.append((trace, label))
    return samples

#   "both" = time-frequency fusion 
#   "time" = only-time-domain 
#   "freq" = only-frequency-domain
# Choose mode: "both"
mode = "both"

##### CuP-test

In [10]:
# Build CuP test set (force label=0)
# Run evaluation + prediction distribution check
final_test_samples = [(trace, 0) for trace, _ in cup_test_samples]
final_test_loader = DataLoader(TraceDataset(final_test_samples), batch_size=32, shuffle=False)

evaluate(tfc_model, classifier, final_test_loader, mode=mode)
predict_class_distribution(tfc_model, classifier, final_test_loader, device)


[Mode=both] Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9880    0.9940       499
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.9880       499
   macro avg     0.5000    0.4940    0.4970       499
weighted avg     1.0000    0.9880    0.9940       499

[Mode=both] Accuracy: 0.9880
Only one true class is present; skipping confusion matrix plotting.


/root/miniconda3/envs/wzyEnv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/miniconda3/envs/wzyEnv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/miniconda3/envs/wzyEnv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Counter({0: 493, 1: 6})

##### CuP-test (G0-shift robustness experiment)

In [11]:
# G0-shift robustness experiment: apply a random conductance offset (delta) to each CuP test trace.
cup_test_samples_shifted = []
cup_deltas = []

random.seed(42)

for t, y in cup_test_samples:
    # Apply a random offset in the range [-0.2, 0.2] to each trace
    delta = random.uniform(-0.2, 0.2)
    t_np = np.asarray(t, dtype=np.float64)
    t_shift = t_np + delta
    cup_test_samples_shifted.append((t_shift, y))
    cup_deltas.append(delta)

print(f"CuP original test trace count: {len(cup_test_samples)}")
print(f"CuP shifted test trace count: {len(cup_test_samples_shifted)}")

# Quick sanity check: print one example trace before/after shifting
idx = 0
k = 10

t_raw, y_raw = cup_test_samples[idx]
t_raw = np.asarray(t_raw, dtype=np.float64)
t_shift, y_shift = cup_test_samples_shifted[idx]

print(f"\n[Example idx={idx}] label={y_raw}, delta={cup_deltas[idx]:.4f}")
print(f"First {k} values of the original trace: {', '.join([f'{x:.4f}' for x in t_raw[:k]])}")
print(f"First {k} values of the shifted trace:   {', '.join([f'{x:.4f}' for x in t_shift[:k]])}")

CuP original test trace count: 499
CuP shifted test trace count: 499

[Example idx=0] label=0, delta=0.0558
First 10 values of the original trace: 0.8456, 0.7293, 0.6785, 0.6412, 0.7213, 0.6338, 0.6326, 0.7966, 0.8076, 0.7421
First 10 values of the shifted trace:   0.9014, 0.7851, 0.7343, 0.6969, 0.7771, 0.6896, 0.6884, 0.8524, 0.8634, 0.7979


In [12]:
# Build CuP test set (force label=0)
# Run evaluation + prediction distribution check
final_test_samples = [(trace, 0) for trace, _ in cup_test_samples_shifted]
final_test_loader = DataLoader(TraceDataset(final_test_samples), batch_size=32, shuffle=False)

evaluate(tfc_model, classifier, final_test_loader, mode=mode)
predict_class_distribution(tfc_model, classifier, final_test_loader, device)


[Mode=both] Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9860    0.9929       499
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.9860       499
   macro avg     0.5000    0.4930    0.4965       499
weighted avg     1.0000    0.9860    0.9929       499

[Mode=both] Accuracy: 0.9860
Only one true class is present; skipping confusion matrix plotting.


/root/miniconda3/envs/wzyEnv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/miniconda3/envs/wzyEnv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/miniconda3/envs/wzyEnv/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Counter({0: 492, 1: 7})